# 04 - Deploy & Predict: Skoring Nasabah dari Data RAW + Uji Berbagai Kasus

Notebook ini mensimulasikan pemakaian pipeline hybrid Layer 1 (ML) +
Layer 2 (policy engine) di produksi, **dimulai murni dari data mentah**
(`retail_customer_profile.csv` + 4 tabel raw lain lewat `NIK`) — bukan dari
`master_dataset.csv` yang sudah di-join (lihat catatan revisi di bawah).

Notebook dibagi jadi 5 bagian:
1. **Bangun fitur dari raw tables** — 1 fungsi reusable, dipakai di semua
   bagian berikutnya (tidak ada logika join yang diduplikasi).
2. **Prediksi 5 applicant nyata** yang diminta.
3. **Uji berbagai edge case** — NIK tidak ditemukan, NIK tidak valid, usia
   di bawah 21, DHN, SLIK Macet, nasabah baru tanpa riwayat SLIK, dan data
   tidak lengkap di tabel lain.
4. **Prediksi 2 applicant BARU** (full profile, belum pernah ada di sistem
   sebelumnya) - data pendukungnya juga dibuat baru di 4 tabel raw lain.
5. **Nasabah existing mengajukan kredit baru** (NIK sudah ada, riwayat SLIK/
   bank/keuangan asli dipakai ulang, cuma nominal pinjaman yang berubah).

## Reuse `utils/agent_pipeline.py` untuk hard-rule check

`utils/risk_ml_pipeline.py::predict_credit_screening()` (STAGE 1) sekarang
memanggil langsung `identity_agent()`, `credit_history_agent()`,
`dhn_agent()` dari `utils/agent_pipeline.py` yang sudah ada — bukan
duplikasi manual. Ini aman karena sudah diverifikasi: hard-reject set dari
`agent_pipeline.py` (Definisi B) **identik 100% (220/220)** dengan
hard-reject set Definisi A (yang dipakai membangun `master_scored.csv` /
melatih model ML) pada seluruh data training. `collateral_agent()` dan
`financial_agent()` dari `agent_pipeline.py` juga dipakai ulang untuk
skor komponen di narasi insight. **Pengecualian:** skor Cashflow untuk
insight **tetap** pakai kalibrasi persentil-90 milik Definisi A, karena
`agent_pipeline.py::cashflow_agent()` terbukti tidak terkalibrasi untuk
skala data ini (dicek: 73% nasabah otomatis skor ≥0.99, membuat flag
"Cashflow lemah" nyaris tidak pernah muncul) — persis masalah yang sudah
didokumentasikan sendiri di `notebooks/02_prepro&eda fix.ipynb` sebagai
alasan kenapa Cashflow direkalibrasi ulang di Definisi A. Ini murni soal
kualitas teks insight, TIDAK memengaruhi `risk_score`/`decision` (itu
sepenuhnya dari model ML + threshold Definisi A).

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
from utils.risk_ml_pipeline import predict_credit_screening, run_hard_rule_agents, _load_artifacts
from utils.feature_builder import build_features_from_raw, load_raw_tables, SLIK_NUM_COLS, BANK_NUM_COLS, FIN_NUM_COLS

model, preprocessor, meta = _load_artifacts()
print(f"Model dipakai: {meta['model_name']}  (dilatih {meta['train_timestamp']})")

profile, slik_full, dhn_full, bank_full, fin_full = load_raw_tables()
print(f"Raw tables loaded: profile={profile.shape}, slik={slik_full.shape}, dhn={dhn_full.shape}, bank={bank_full.shape}, fin={fin_full.shape}")

Model dipakai: XGBoost  (dilatih 20260825_163612)
Raw tables loaded: profile=(3000, 38), slik=(4416, 11), dhn=(3000, 5), bank=(4052, 13), fin=(6000, 8)


## 1. Fungsi Reusable: Bangun Fitur dari Raw Tables (by `application_id`)

`build_features_from_raw()` sekarang tinggal di `utils/feature_builder.py`
(bukan didefinisikan lokal di sini lagi) — supaya `pages/3_Simulasi.py`
di dashboard Streamlit bisa memanggil fungsi yang SAMA PERSIS, bukan
duplikasi ketiga kalinya. Join/agregasinya identik dengan
`notebooks/02_prepro&eda fix.ipynb`. Tiga penyesuaian robustness ada di
sana (tidak ada di notebook 02 asli, karena di sana semua NIK dijamin
lengkap secara desain generator):

- **`application_id` tidak ditemukan** → `ValueError` yang jelas, bukan
  hasil kosong/salah diam-diam.
- **NIK tidak punya baris di `bank_account.csv` atau `laporan_keuangan.csv`**
  (skenario yang tidak pernah terjadi di data sintetis ini, tapi *bisa*
  terjadi di produksi nyata kalau ada keterlambatan sinkronisasi data) →
  di-`fillna(0)`, sama seperti perlakuan kolom SLIK, alih-alih membiarkan
  `NaN` merambat diam-diam ke preprocessor/model.
- **`status_dhn` tidak ketemu di `dhn.csv` sama sekali** -> di-`fillna("Tidak")`, bukan dibiarkan `NaN` (yang kebetulan juga aman lewat `_yes(NaN)==False`, tapi sekarang eksplisit).

In [2]:
# build_features_from_raw() dipindah ke utils/feature_builder.py -
# dipakai bersama oleh notebook ini DAN pages/3_Simulasi.py, supaya
# logika join/agregasinya cuma ada di satu tempat.
print("Fungsi build_features_from_raw() diimpor dari utils/feature_builder.py")

Fungsi build_features_from_raw() diimpor dari utils/feature_builder.py


## 2. Prediksi 5 Applicant Nyata (dari `retail_customer_profile.csv`)

In [3]:
TARGET_IDS = ["APP202600001", "APP202600002", "APP202600003", "APP202600004", "APP202600005"]
master_row = build_features_from_raw(TARGET_IDS, profile, slik_full, dhn_full, bank_full, fin_full)
print(f"Fitur berhasil dibangun murni dari raw tables: {master_row.shape}")
master_row[["application_id", "status_dhn", "slik_worst_collectability", "slik_has_credit_history",
            "revenue_growth_pct", "bank_best_avg_balance_6m"]]

Fitur berhasil dibangun murni dari raw tables: (5, 70)


,application_id,status_dhn,slik_worst_collectability,slik_has_credit_history,revenue_growth_pct,bank_best_avg_balance_6m
0,APP202600001,Tidak,1,1,0.0809,19800614
1,APP202600002,Tidak,2,1,-0.0292,4014256
2,APP202600003,Tidak,4,1,-0.1476,14489012
3,APP202600004,Tidak,2,1,-0.0377,8551321
4,APP202600005,Tidak,2,1,0.2762,10845679


In [4]:
results = []
for _, row in master_row.iterrows():
    pred = predict_credit_screening(row.to_dict())
    pred.update({"application_id": row["application_id"], "company_name": row["company_name"], "label_asli": row["label"]})
    results.append(pred)

summary = pd.DataFrame([
    {
        "application_id": r["application_id"], "company_name": r["company_name"],
        "Decision": r["decision"], "Risk Score": r["risk_score"], "Zone": r["zone"],
        "Jenis Kredit": r["jenis_kredit_rekomendasi"], "Nominal Disetujui": r["nominal_disetujui"],
        "Jangka Waktu (bulan)": r["jangka_waktu_bulan"], "Bunga (%)": r["bunga_persen"],
        "Label Asli": r["label_asli"],
    }
    for r in results
])
summary

,application_id,company_name,Decision,Risk Score,Zone,Jenis Kredit,Nominal Disetujui,Jangka Waktu (bulan),Bunga (%),Label Asli
0,APP202600001,UD Santoso Abadi,Layak,0.824,Hijau,KI,300000000,36,9.5,Diterima
1,APP202600002,UD Wijaya Mandiri,Layak,0.713,Hijau,KMK,75000000,12,9.5,Diterima
2,APP202600003,UD Kusuma Sejahtera,Perlu Review Ulang,0.505,Kuning,KMK,150000000,12,12.0,Ditolak
3,APP202600004,UD Susanto Makmur,Layak,0.708,Hijau,KI,200000000,36,9.5,Diterima
4,APP202600005,CV Wijaya Sejahtera,Layak,0.834,Hijau,KI,750000000,36,9.5,Diterima


### Sanity check — bandingkan hasil rebuild vs `master_dataset.csv`

Murni untuk *membuktikan* proses join manual di atas benar — bukan bagian
dari jalur deploy (di produksi nyata `master_dataset.csv` tidak ada).

In [5]:
check_cols = ["status_dhn", "slik_worst_collectability", "revenue_growth_pct", "profit_margin_2025",
              "bank_best_avg_balance_6m", "bank_any_dormant", "dsr_capped"]
master_dataset = pd.read_csv("../data/processed/master_dataset.csv", dtype={"NIK": str})
reference = master_dataset[master_dataset["application_id"].isin(TARGET_IDS)].set_index("application_id")[check_cols]
rebuilt = master_row.set_index("application_id")[check_cols].reindex(reference.index)

# Perbandingan numeric-aware (bukan string): master_dataset.csv menyimpan
# kolom seperti slik_worst_collectability sebagai float64 (karena di
# full 3.000-baris dataset ada NaN yang di-fillna, memaksa dtype float utk
# SELURUH kolom), sedangkan rebuild 5-baris ini kebetulan tidak ada NaN sama
# sekali -> tetap int64. Nilainya identik (1.0 == 1), cuma representasi
# dtype yang beda - jadi wajar dibandingkan sebagai angka, bukan teks.
mismatches = {}
for col in check_cols:
    ref_col, reb_col = reference[col], rebuilt[col]
    try:
        same = np.isclose(ref_col.astype(float), reb_col.astype(float), equal_nan=True)
    except (TypeError, ValueError):
        same = (ref_col.astype(str) == reb_col.astype(str))
    if not all(same):
        mismatches[col] = pd.DataFrame({"master_dataset.csv": ref_col, "rebuilt": reb_col})

print("Semua kolom cocok dengan master_dataset.csv?", len(mismatches) == 0)
for col, diff_df in mismatches.items():
    print(f"\nMismatch pada kolom '{col}':")
    print(diff_df)

Semua kolom cocok dengan master_dataset.csv? True


## 3. Uji Berbagai Kasus (Edge Cases)

Tabel kasus yang diuji — mencakup kegagalan lookup data, hard-rule
kill-switch, dan skenario data tidak lengkap:

| # | Kasus | Ekspektasi |
|---|---|---|
| 1 | `application_id` tidak ditemukan | `ValueError` yang jelas dari `build_features_from_raw()`, bukan crash tidak jelas |
| 2 | Format NIK tidak valid (bukan 16 digit) | Hard-reject STAGE 1 (`identity_agent`) |
| 3 | Usia pemohon di bawah 21 tahun | Hard-reject STAGE 1 (`identity_agent`) |
| 4 | Terdaftar di DHN (Daftar Hitam Nasional) | Hard-reject STAGE 1 (`dhn_agent`) |
| 5 | SLIK Macet (kolektibilitas 5) | Hard-reject STAGE 1 (`credit_history_agent`) |
| 6 | Nasabah baru, belum ada riwayat SLIK sama sekali | **Bukan** hard-reject — lanjut ke ML (STAGE 2) dengan default netral |
| 7 | Data tidak lengkap (NIK tidak ada di `bank_account`/`laporan_keuangan`) | Tidak crash — fitur di-fillna 0, tetap lanjut ke ML |

In [6]:
print("=" * 70)
print("KASUS 1 — application_id tidak ditemukan")
print("=" * 70)
try:
    build_features_from_raw("APP209999999", profile, slik_full, dhn_full, bank_full, fin_full)
except ValueError as e:
    print(f"Ditangani dengan benar -> ValueError: {e}")

KASUS 1 — application_id tidak ditemukan
Ditangani dengan benar -> ValueError: application_id tidak ditemukan di profile: ['APP209999999']


In [7]:
print("=" * 70)
print("KASUS 2 — Format NIK tidak valid (13 digit, bukan 16)")
print("=" * 70)
base = build_features_from_raw("APP202600001", profile, slik_full, dhn_full, bank_full, fin_full).iloc[0].to_dict()
kasus2 = dict(base)
kasus2["NIK"] = "123456789"  # < 16 digit
hasil2 = predict_credit_screening(kasus2)
print(f"Decision: {hasil2['decision']}  |  risk_score: {hasil2['risk_score']}")
print(f"Insight : {hasil2['insight']}")
assert hasil2["decision"] == "Tidak Layak" and hasil2["risk_score"] is None

KASUS 2 — Format NIK tidak valid (13 digit, bukan 16)
Decision: Tidak Layak  |  risk_score: None
Insight : Tidak layak karena nik tidak valid (harus 16 digit angka): '123456789'.


In [8]:
print("=" * 70)
print("KASUS 3 — Usia pemohon di bawah 21 tahun")
print("=" * 70)
kasus3 = dict(base)
kasus3["owner_age"] = 19
hasil3 = predict_credit_screening(kasus3)
print(f"Decision: {hasil3['decision']}  |  risk_score: {hasil3['risk_score']}")
print(f"Insight : {hasil3['insight']}")
assert hasil3["decision"] == "Tidak Layak" and hasil3["risk_score"] is None

KASUS 3 — Usia pemohon di bawah 21 tahun
Decision: Tidak Layak  |  risk_score: None
Insight : Tidak layak karena usia pemohon 19 di luar rentang layak 21-65 tahun.


In [9]:
print("=" * 70)
print("KASUS 4 — Terdaftar di DHN (contoh nyata: APP202600141)")
print("=" * 70)
row4 = build_features_from_raw("APP202600141", profile, slik_full, dhn_full, bank_full, fin_full).iloc[0]
print(f"status_dhn (raw) = {row4['status_dhn']}")
hasil4 = predict_credit_screening(row4.to_dict())
print(f"Decision: {hasil4['decision']}  |  risk_score: {hasil4['risk_score']}")
print(f"Insight : {hasil4['insight']}")
assert hasil4["decision"] == "Tidak Layak" and hasil4["risk_score"] is None

KASUS 4 — Terdaftar di DHN (contoh nyata: APP202600141)
status_dhn (raw) = Ya
Decision: Tidak Layak  |  risk_score: None
Insight : Tidak layak karena nasabah terdaftar di Daftar Hitam Nasional (Laporan pihak ketiga terkait sengketa usaha).


In [10]:
print("=" * 70)
print("KASUS 5 — SLIK Macet, kolektibilitas 5 (contoh nyata: APP202600129)")
print("=" * 70)
row5 = build_features_from_raw("APP202600129", profile, slik_full, dhn_full, bank_full, fin_full).iloc[0]
print(f"slik_worst_collectability (raw) = {row5['slik_worst_collectability']}")
hasil5 = predict_credit_screening(row5.to_dict())
print(f"Decision: {hasil5['decision']}  |  risk_score: {hasil5['risk_score']}")
print(f"Insight : {hasil5['insight']}")
assert hasil5["decision"] == "Tidak Layak" and hasil5["risk_score"] is None

KASUS 5 — SLIK Macet, kolektibilitas 5 (contoh nyata: APP202600129)
slik_worst_collectability (raw) = 5
Decision: Tidak Layak  |  risk_score: None
Insight : Tidak layak karena memiliki riwayat kredit Macet pada SLIK.


In [11]:
print("=" * 70)
print("KASUS 6 — Nasabah baru, belum ada riwayat SLIK sama sekali (contoh nyata: APP202600009)")
print("=" * 70)
row6 = build_features_from_raw("APP202600009", profile, slik_full, dhn_full, bank_full, fin_full).iloc[0]
print(f"slik_has_credit_history (raw, setelah fillna) = {row6['slik_has_credit_history']}")
print(f"slik_worst_collectability (raw, setelah fillna) = {row6['slik_worst_collectability']}")
agents6 = run_hard_rule_agents(row6.to_dict())
print(f"credit_history_agent -> score={agents6['credit_history']['score']}, hard_reject={agents6['credit_history']['hard_reject']}")
hasil6 = predict_credit_screening(row6.to_dict())
print(f"Decision: {hasil6['decision']}  |  risk_score: {hasil6['risk_score']}  (BUKAN hard-reject -> model ML tetap dipanggil)")
print(f"Insight : {hasil6['insight']}")
assert hasil6["risk_score"] is not None, "Nasabah baru tanpa riwayat SLIK seharusnya TETAP lewat ke ML, bukan hard-reject!" 

KASUS 6 — Nasabah baru, belum ada riwayat SLIK sama sekali (contoh nyata: APP202600009)
slik_has_credit_history (raw, setelah fillna) = 0
slik_worst_collectability (raw, setelah fillna) = 0.0
credit_history_agent -> score=0.6, hard_reject=False
Decision: Layak Bersyarat  |  risk_score: 0.662  (BUKAN hard-reject -> model ML tetap dipanggil)
Insight : Layak bersyarat karena Cashflow — disarankan tambahan agunan/penjamin atau plafon diturunkan.


In [12]:
print("=" * 70)
print("KASUS 7 — Data tidak lengkap: NIK sengaja dihilangkan dari bank_account & laporan_keuangan")
print("=" * 70)
target_nik_7 = profile[profile["application_id"] == "APP202600001"]["NIK"].iloc[0]
bank_incomplete = bank_full[bank_full["NIK"] != target_nik_7]   # simulasi: NIK ini "hilang" dari tabel rekening
fin_incomplete = fin_full[fin_full["NIK"] != target_nik_7]      # dan dari tabel laporan keuangan

row7 = build_features_from_raw("APP202600001", profile, slik_full, dhn_full, bank=bank_incomplete, fin=fin_incomplete).iloc[0]
print(f"bank_best_avg_balance_6m (setelah fillna, harusnya 0) = {row7['bank_best_avg_balance_6m']}")
print(f"revenue_2024 (setelah fillna, harusnya 0)             = {row7['revenue_2024']}")
assert not row7[BANK_NUM_COLS + FIN_NUM_COLS].isna().any(), "Masih ada NaN yang lolos!"

hasil7 = predict_credit_screening(row7.to_dict())
print(f"\nTidak crash. Decision: {hasil7['decision']}  |  risk_score: {hasil7['risk_score']}")
print(f"Insight : {hasil7['insight']}")
print("\n(Dibandingkan dgn data lengkap, skor turun drastis karena finansial/cashflow dianggap 0 - ")
print(" masuk akal: sistem menilai 'tidak ada data' seolah 'tidak ada aktivitas', BUKAN error.)")

KASUS 7 — Data tidak lengkap: NIK sengaja dihilangkan dari bank_account & laporan_keuangan
bank_best_avg_balance_6m (setelah fillna, harusnya 0) = 0.0
revenue_2024 (setelah fillna, harusnya 0)             = 0.0



Tidak crash. Decision: Layak  |  risk_score: 0.757
Insight : Layak tapi Financial, Cashflow tergolong lemah (skor di bawah 0.5) — disarankan tetap dimonitor meski keputusan akhir disetujui.

(Dibandingkan dgn data lengkap, skor turun drastis karena finansial/cashflow dianggap 0 - 
 masuk akal: sistem menilai 'tidak ada data' seolah 'tidak ada aktivitas', BUKAN error.)


### Ringkasan hasil ke-7 kasus

In [13]:
edge_summary = pd.DataFrame([
    {"Kasus": "1. application_id tidak ditemukan", "Hasil": "ValueError (ditangani, tidak crash diam-diam)", "risk_score": "-"},
    {"Kasus": "2. NIK tidak valid (<16 digit)", "Hasil": hasil2["decision"], "risk_score": hasil2["risk_score"]},
    {"Kasus": "3. Usia < 21 tahun", "Hasil": hasil3["decision"], "risk_score": hasil3["risk_score"]},
    {"Kasus": "4. DHN blacklist (APP202600141)", "Hasil": hasil4["decision"], "risk_score": hasil4["risk_score"]},
    {"Kasus": "5. SLIK Macet (APP202600129)", "Hasil": hasil5["decision"], "risk_score": hasil5["risk_score"]},
    {"Kasus": "6. Nasabah baru tanpa riwayat SLIK (APP202600009)", "Hasil": hasil6["decision"], "risk_score": hasil6["risk_score"]},
    {"Kasus": "7. Data tidak lengkap (bank+keuangan hilang)", "Hasil": hasil7["decision"], "risk_score": hasil7["risk_score"]},
])
edge_summary

,Kasus,Hasil,risk_score
0,1. application_id tidak ditemukan,"ValueError (ditangani, tidak crash diam-diam)",-
1,2. NIK tidak valid (<16 digit),Tidak Layak,None
2,3. Usia < 21 tahun,Tidak Layak,None
3,4. DHN blacklist (APP202600141),Tidak Layak,None
4,5. SLIK Macet (APP202600129),Tidak Layak,None
5,6. Nasabah baru tanpa riwayat SLIK (APP202600009),Layak Bersyarat,0.662
6,7. Data tidak lengkap (bank+keuangan hilang),Layak,0.757


## 4. Prediksi 2 Applicant BARU (Full Profile, Belum Pernah Ada di Sistem)

Dua applicant benar-benar baru — `application_id` dan `NIK` tidak ada di
`data/raw/` mana pun. Semua 38 kolom `retail_customer_profile.csv` diisi
lengkap, PLUS data pendukung baru di 4 tabel raw lain (SLIK, DHN, rekening
bank, laporan keuangan) — mensimulasikan pengajuan kredit pertama kali,
di mana data-data ini didapat lewat proses aplikasi (SLIK ditarik dari
OJK, laporan keuangan dari calon debitur, rekening koran, dst.), bukan
sudah tersedia di sistem sebelumnya. `build_features_from_raw()` dipanggil
apa adanya dengan tabel-tabel baru ini di-passing sebagai parameter —
tidak ada satu baris pun di `data/raw/*.csv` yang diubah.

**Profil yang dibuat (sengaja dibedakan untuk lihat rentang hasil):**

| | Applicant 1 (`APP202699001`) | Applicant 2 (`APP202699002`) |
|---|---|---|
| Usaha | CV Mitra Boga Sejahtera — Kuliner (Restoran), 9 tahun jalan | PT Karya Teknik Nusantara — Jasa (Konstruksi Kecil), 2 tahun jalan |
| SLIK | 1 fasilitas, Lancar | 2 fasilitas, salah satunya DPK (kolektibilitas 2) |
| DSR estimasi | 0.9 | 2.1 (lebih berat) |
| Growth omset | +20% YoY | +3.8% YoY (stagnan) |

**Catatan `eligibility_score` & `label`:** kedua kolom ini adalah output
formula generator internal (`compute_label_score`, notebook 01) yang
dipakai membuat ground truth SINTETIS saat data training dibuat — bukan
sesuatu yang tersedia untuk applicant sungguhan. Untuk 2 applicant baru
ini kedua kolom sengaja dikosongkan (`NaN`), bukan direka-reka, karena
`predict_credit_screening()` memang tidak pernah membacanya (sudah
dikecualikan sebagai leakage sejak `03_ml_risk_scoring.ipynb`).

> **Update:** setelah `identity_agent()` di `utils/agent_pipeline.py` diperketat (NIK sekarang juga wajib terdaftar di `data/raw/dukcapil.csv`, tidak cukup 16 digit angka saja), kedua NIK karangan di bawah **otomatis hard-reject di STAGE 1** — lihat penjelasan setelah tabel hasil.

In [14]:
new_profile = pd.DataFrame([
    {
        "application_id": "APP202699001", "NIK": "3276014205830091", "cif_number": "CIF1099001",
        "application_date": "2026-08-20", "customer_type": "UMKM", "company_name": "CV Mitra Boga Sejahtera",
        "legal_entity": "CV", "owner_name": "Hendra Kusnadi", "owner_gender": "L", "owner_age": 43,
        "owner_marital_status": "Menikah", "owner_education": "S1",
        "province": "Jawa Barat", "city": "Depok", "district": "Beji", "region": "Region 3",
        "branch_name": "KCP Depok Margonda", "industry": "Kuliner", "sub_industry": "Restoran",
        "business_age_year": 9, "employee_count": 15, "monthly_turnover_est": 120_000_000,
        "transaction_frequency_monthly": 180, "loan_requested": 250_000_000,
        "collateral_type": "Ruko", "collateral_location": "Margonda, Depok",
        "collateral_province": "Jawa Barat", "collateral_city": "Depok", "collateral_size_m2": 150.0,
        "collateral_market_value": 1_800_000_000, "collateral_liquidation_value": 1_440_000_000,
        "collateral_ratio": round(1_800_000_000 / 250_000_000, 2), "certificate_type": "SHM",
        "ownership_match": "Ya", "estimated_dsr": 0.9,
        "eligibility_score": np.nan, "label": np.nan, "rm_id": "RM0013",
    },
    {
        "application_id": "APP202699002", "NIK": "3175024508910077", "cif_number": "CIF1099002",
        "application_date": "2026-08-21", "customer_type": "UMKM", "company_name": "PT Karya Teknik Nusantara",
        "legal_entity": "PT", "owner_name": "Siti Rahmawati", "owner_gender": "P", "owner_age": 31,
        "owner_marital_status": "Menikah", "owner_education": "D3",
        "province": "DKI Jakarta", "city": "Jakarta Utara", "district": "Kelapa Gading", "region": "Region 2",
        "branch_name": "KCP Kelapa Gading", "industry": "Jasa", "sub_industry": "Jasa Konstruksi Kecil",
        "business_age_year": 2, "employee_count": 6, "monthly_turnover_est": 45_000_000,
        "transaction_frequency_monthly": 60, "loan_requested": 180_000_000,
        "collateral_type": "Rumah", "collateral_location": "Kelapa Gading, Jakarta Utara",
        "collateral_province": "DKI Jakarta", "collateral_city": "Jakarta Utara", "collateral_size_m2": 90.0,
        "collateral_market_value": 950_000_000, "collateral_liquidation_value": 760_000_000,
        "collateral_ratio": round(950_000_000 / 180_000_000, 2), "certificate_type": "HGB",
        "ownership_match": "Ya", "estimated_dsr": 2.1,
        "eligibility_score": np.nan, "label": np.nan, "rm_id": "RM0005",
    },
])
new_profile["NIK"] = new_profile["NIK"].astype(str)
new_profile = new_profile[profile.columns.tolist()]  # pastikan urutan & kelengkapan kolom PERSIS sama dgn retail_customer_profile.csv

assert not new_profile["application_id"].isin(profile["application_id"]).any(), "application_id baru bentrok dgn yang sudah ada!"
assert not new_profile["NIK"].isin(profile["NIK"]).any(), "NIK baru bentrok dgn yang sudah ada!"
print(f"2 applicant baru dibuat, {new_profile.shape[1]} kolom (sama persis dgn retail_customer_profile.csv)")
new_profile

2 applicant baru dibuat, 38 kolom (sama persis dgn retail_customer_profile.csv)


,application_id,NIK,cif_number,application_date,customer_type,company_name,legal_entity,owner_name,owner_gender,owner_age,...,collateral_size_m2,collateral_market_value,collateral_liquidation_value,collateral_ratio,certificate_type,ownership_match,estimated_dsr,eligibility_score,label,rm_id
0,APP202699001,3276014205830091,CIF1099001,2026-08-20,UMKM,CV Mitra Boga Sejahtera,CV,Hendra Kusnadi,L,43,...,150.0,1800000000,1440000000,7.20,SHM,Ya,0.9,NaN,NaN,RM0013
1,APP202699002,3175024508910077,CIF1099002,2026-08-21,UMKM,PT Karya Teknik Nusantara,PT,Siti Rahmawati,P,31,...,90.0,950000000,760000000,5.28,HGB,Ya,2.1,NaN,NaN,RM0005


### Data pendukung baru di 4 tabel raw lain (by `NIK`)

In [15]:
new_dhn = pd.DataFrame([
    {"dhn_id": "DHN_NEW001", "NIK": "3276014205830091", "status_dhn": "Tidak", "alasan": None, "tanggal_input": "2026-08-15"},
    {"dhn_id": "DHN_NEW002", "NIK": "3175024508910077", "status_dhn": "Tidak", "alasan": None, "tanggal_input": "2026-08-16"},
])

new_slik = pd.DataFrame([
    {"slik_record_id": "SLK_NEW001", "NIK": "3276014205830091", "inquiry_date": "2026-08-10",
     "bank_name": "Bank Mandiri", "loan_type": "KMK", "plafond": 80_000_000, "outstanding_balance": 30_000_000,
     "installment_amount": 3_000_000, "tenor_month": 24, "collectability": 1, "collectability_label": "Lancar"},
    {"slik_record_id": "SLK_NEW002", "NIK": "3175024508910077", "inquiry_date": "2026-08-11",
     "bank_name": "Bank BRI", "loan_type": "KK", "plafond": 25_000_000, "outstanding_balance": 18_000_000,
     "installment_amount": 2_200_000, "tenor_month": 18, "collectability": 2, "collectability_label": "Dalam Perhatian Khusus (DPK)"},
    {"slik_record_id": "SLK_NEW003", "NIK": "3175024508910077", "inquiry_date": "2026-08-11",
     "bank_name": "Bank Danamon", "loan_type": "KMK", "plafond": 15_000_000, "outstanding_balance": 6_000_000,
     "installment_amount": 900_000, "tenor_month": 12, "collectability": 1, "collectability_label": "Lancar"},
])

new_bank = pd.DataFrame([
    {"account_id": "ACC_NEW001", "NIK": "3276014205830091", "account_number": "9010000001",
     "bank_name": "Bank BCA", "account_type": "Giro", "account_status": "Aktif", "opened_date": "2018-03-01",
     "average_balance_6m": 7_200_000_000, "average_monthly_credit": 130_000_000, "average_monthly_debit": 118_000_000,
     "transaction_frequency_monthly": 180, "overdraft_count_6m": 0, "current_balance": 7_500_000_000},
    {"account_id": "ACC_NEW002", "NIK": "3175024508910077", "account_number": "9010000002",
     "bank_name": "Bank BNI", "account_type": "Tabungan", "account_status": "Aktif", "opened_date": "2023-05-15",
     "average_balance_6m": 675_000_000, "average_monthly_credit": 48_000_000, "average_monthly_debit": 46_000_000,
     "transaction_frequency_monthly": 60, "overdraft_count_6m": 1, "current_balance": 620_000_000},
])

new_fin = pd.DataFrame([
    {"laporan_id": "FIN_NEW001", "NIK": "3276014205830091", "year": 2024, "revenue": 1_050_000_000,
     "net_profit": 110_000_000, "total_asset": 780_000_000, "total_liability": 260_000_000, "operating_cashflow": 95_000_000},
    {"laporan_id": "FIN_NEW002", "NIK": "3276014205830091", "year": 2025, "revenue": 1_260_000_000,
     "net_profit": 145_000_000, "total_asset": 900_000_000, "total_liability": 230_000_000, "operating_cashflow": 128_000_000},
    {"laporan_id": "FIN_NEW003", "NIK": "3175024508910077", "year": 2024, "revenue": 520_000_000,
     "net_profit": 28_000_000, "total_asset": 310_000_000, "total_liability": 190_000_000, "operating_cashflow": 32_000_000},
    {"laporan_id": "FIN_NEW004", "NIK": "3175024508910077", "year": 2025, "revenue": 540_000_000,
     "net_profit": 30_000_000, "total_asset": 330_000_000, "total_liability": 205_000_000, "operating_cashflow": 35_000_000},
])

print(f"new_dhn={new_dhn.shape}, new_slik={new_slik.shape}, new_bank={new_bank.shape}, new_fin={new_fin.shape}")

new_dhn=(2, 5), new_slik=(3, 11), new_bank=(2, 13), new_fin=(4, 8)


### Bangun fitur (fungsi yang sama, `profile`/`slik`/`dhn`/`bank`/`fin` diganti ke data baru) + prediksi

In [16]:
NEW_IDS = ["APP202699001", "APP202699002"]
new_features = build_features_from_raw(NEW_IDS, profile=new_profile, slik=new_slik, dhn=new_dhn, bank=new_bank, fin=new_fin)

new_features[["application_id", "status_dhn", "slik_worst_collectability", "slik_n_banks",
              "revenue_growth_pct", "bank_best_avg_balance_6m", "collateral_ratio", "dsr_capped"]]

,application_id,status_dhn,slik_worst_collectability,slik_n_banks,revenue_growth_pct,bank_best_avg_balance_6m,collateral_ratio,dsr_capped
0,APP202699001,Tidak,1,1,0.2000,7200000000,7.20,0.9
1,APP202699002,Tidak,2,2,0.0385,675000000,5.28,2.1


In [17]:
new_results = []
for _, row in new_features.iterrows():
    pred = predict_credit_screening(row.to_dict())
    pred.update({"application_id": row["application_id"], "company_name": row["company_name"]})
    new_results.append(pred)

for r in new_results:
    print("=" * 70)
    print(f"application_id : {r['application_id']}  ({r['company_name']})")
    print(f"Decision                       : {r['decision']}")
    print(f"Risk Score / Eligibility Score : {r['risk_score']}")
    print(f"Zone                           : {r['zone']}")
    print(f"Jenis Kredit                   : {r['jenis_kredit_rekomendasi']}")
    print(f"Nominal Kredit Disetujui       : {r['nominal_disetujui']:,}".replace(",", "."))
    print(f"Jangka Waktu                   : {r['jangka_waktu_bulan']} bulan")
    print(f"Bunga                          : {r['bunga_persen']}% p.a." if r["bunga_persen"] is not None else "Bunga                          : -")
    print(f"Insight                        : {r['insight']}")
print("=" * 70)

application_id : APP202699001  (CV Mitra Boga Sejahtera)
Decision                       : Tidak Layak
Risk Score / Eligibility Score : None
Zone                           : Merah
Jenis Kredit                   : -
Nominal Kredit Disetujui       : 0
Jangka Waktu                   : 0 bulan
Bunga                          : -
Insight                        : Tidak layak karena nik '3276014205830091' tidak ditemukan di data dukcapil.
application_id : APP202699002  (PT Karya Teknik Nusantara)
Decision                       : Tidak Layak
Risk Score / Eligibility Score : None
Zone                           : Merah
Jenis Kredit                   : -
Nominal Kredit Disetujui       : 0
Jangka Waktu                   : 0 bulan
Bunga                          : -
Insight                        : Tidak layak karena nik '3175024508910077' tidak ditemukan di data dukcapil.


### Tabel ringkasan — 2 applicant baru

In [18]:
new_summary = pd.DataFrame([
    {
        "application_id": r["application_id"], "company_name": r["company_name"],
        "Decision (Eligibility Recommendation)": r["decision"],
        "Risk Score / Eligibility Score": r["risk_score"], "Zone": r["zone"],
        "Jenis Kredit": r["jenis_kredit_rekomendasi"], "Nominal Disetujui": r["nominal_disetujui"],
        "Jangka Waktu (bulan)": r["jangka_waktu_bulan"], "Bunga (% p.a.)": r["bunga_persen"],
    }
    for r in new_results
])
new_summary

,application_id,company_name,Decision (Eligibility Recommendation),Risk Score / Eligibility Score,Zone,Jenis Kredit,Nominal Disetujui,Jangka Waktu (bulan),Bunga (% p.a.)
0,APP202699001,CV Mitra Boga Sejahtera,Tidak Layak,None,Merah,-,0,0,None
1,APP202699002,PT Karya Teknik Nusantara,Tidak Layak,None,Merah,-,0,0,None


### Kenapa keduanya sekarang `Tidak Layak`?

`identity_agent()` (STAGE 1, di `utils/agent_pipeline.py`) sekarang mewajibkan NIK
terdaftar di `data/raw/dukcapil.csv` — bukan cuma format 16 digit. NIK
`3276014205830091` dan `3175024508910077` di atas murni karangan untuk demo ini,
jadi **keduanya tidak pernah ada di `dukcapil.csv`** (yang di dataset ini adalah
himpunan tertutup, persis 3.000 NIK yang sama dengan seluruh pengajuan yang ada —
bukan simulasi seluruh penduduk terdaftar). Hasilnya, kedua applicant sekarang
hard-reject di STAGE 1 sebelum model ML sempat dipanggil sama sekali, terlepas dari
seberapa bagus profil finansial/agunan yang dibuat.

**Ini konsisten dengan cara kerja KYC di dunia nyata** — NIK yang tidak bisa
diverifikasi ke Dukcapil memang seharusnya ditolak otomatis, apa pun profil
finansialnya. Untuk mendemokan skenario "nasabah baru yang identitasnya valid tapi
belum punya riwayat kredit" (bukan "orang yang tidak terdaftar sama sekali"), lihat
Bagian 3 Kasus 6 di atas (`APP202600009`) — NIK asli, terverifikasi Dukcapil, tapi
`slik_has_credit_history=0` — kasus itu **tetap lolos ke ML**, bukan hard-reject,
karena "belum pernah kredit" secara semantik berbeda dari "identitasnya tidak
dikenali".

## 5. Nasabah Existing Mengajukan Kredit BARU (Nominal Lebih Besar)

Skenario berbeda dari Bagian 4: kali ini **NIK-nya sudah ada** di sistem
(`3175015609750023` dan `3671016012760030`, masing-masing sudah punya
pengajuan sebelumnya — `APP202600023` dan `APP202600030`) dan sudah punya
riwayat nyata di SLIK/rekening bank/laporan keuangan. Sekarang mereka
mengajukan kredit **baru** (`application_id` baru) dengan `loan_requested`
lebih besar — data profil/identitas/agunan lain **tetap sama** (metadata
disalin apa adanya dari pengajuan lama), cuma nominal pinjaman & rasio
agunan yang berubah.

Karena NIK-nya sudah ada, SLIK/DHN/rekening/laporan keuangan **tidak perlu
dibuat baru** — cukup panggil `build_features_from_raw()` dengan
`slik=slik_full, dhn=dhn_full, bank=bank_full, fin=fin_full` (tabel raw
ASLI, bukan tabel fabrikasi seperti di Bagian 4) — riwayat nyata nasabah
otomatis ikut ter-agregasi lewat `NIK` yang sama.

| | Pengajuan LAMA | Pengajuan BARU |
|---|---|---|
| NIK `3175015609750023` (UD Susanto Mandiri) | `APP202600023`, pinjaman 750 juta, rasio agunan 25.0x | `APP202699003`, pinjaman **1,2 M** (+60%), rasio agunan turun ke ~15.6x |
| NIK `3671016012760030` (PT Kusuma Makmur) | `APP202600030`, pinjaman 750 juta, rasio agunan 2.51x | `APP202699004`, pinjaman **1,1 M** (+47%), rasio agunan turun ke ~1.7x (makin ketat) |

In [19]:
REPEAT_CASES = [
    {"old_application_id": "APP202600023", "new_application_id": "APP202699003", "new_loan_requested": 1_200_000_000},
    {"old_application_id": "APP202600030", "new_application_id": "APP202699004", "new_loan_requested": 1_100_000_000},
]

repeat_rows = []
for case in REPEAT_CASES:
    old_row = profile[profile["application_id"] == case["old_application_id"]].iloc[0].to_dict()
    new_row = dict(old_row)  # salin SEMUA metadata (NIK, cif_number, owner, agunan, dst.) apa adanya
    new_row["application_id"] = case["new_application_id"]
    new_row["application_date"] = "2026-08-25"
    new_row["loan_requested"] = case["new_loan_requested"]
    new_row["collateral_ratio"] = round(new_row["collateral_market_value"] / case["new_loan_requested"], 2)
    # eligibility_score/label lama milik PENGAJUAN LAMA (nominal beda) - tidak valid utk pengajuan baru,
    # dan memang tidak pernah dibaca predict_credit_screening() (leakage), jadi dikosongkan apa adanya.
    new_row["eligibility_score"] = np.nan
    new_row["label"] = np.nan
    repeat_rows.append(new_row)

repeat_profile = pd.DataFrame(repeat_rows)[profile.columns.tolist()]

assert not repeat_profile["application_id"].isin(profile["application_id"]).any(), "application_id baru bentrok!"
assert repeat_profile["NIK"].isin(profile["NIK"]).all(), "NIK seharusnya sudah eksis di sistem!"
print("cif_number tetap sama dgn pengajuan lama (nasabah yang sama):")
print(repeat_profile[["application_id", "NIK", "cif_number", "company_name", "loan_requested", "collateral_ratio"]])

cif_number tetap sama dgn pengajuan lama (nasabah yang sama):
  application_id               NIK  cif_number        company_name  \
0   APP202699003  3175015609750023  CIF1000023  UD Susanto Mandiri   
1   APP202699004  3671016012760030  CIF1000030    PT Kusuma Makmur   

   loan_requested  collateral_ratio  
0      1200000000             15.62  
1      1100000000              1.71  


### Bangun fitur — pakai tabel raw ASLI (`slik_full`, `dhn_full`, `bank_full`, `fin_full`), bukan fabrikasi

In [20]:
NEW_REPEAT_IDS = [c["new_application_id"] for c in REPEAT_CASES]
repeat_features = build_features_from_raw(NEW_REPEAT_IDS, profile=repeat_profile,
                                           slik=slik_full, dhn=dhn_full, bank=bank_full, fin=fin_full)

repeat_features[["application_id", "NIK", "status_dhn", "slik_worst_collectability", "slik_n_banks",
                  "revenue_growth_pct", "bank_best_avg_balance_6m", "loan_requested", "collateral_ratio"]]

,application_id,NIK,status_dhn,slik_worst_collectability,slik_n_banks,revenue_growth_pct,bank_best_avg_balance_6m,loan_requested,collateral_ratio
0,APP202699003,3175015609750023,Tidak,3,2,-0.1844,11638176,1200000000,15.62
1,APP202699004,3671016012760030,Tidak,1,2,0.0907,11035460,1100000000,1.71


### Bandingkan: Pengajuan Lama vs Pengajuan Baru (nominal lebih besar)

Riwayat SLIK/rekening/keuangan-nya **identik** (nasabah yang sama) — yang
berubah murni `loan_requested` & `collateral_ratio` turunannya, jadi
selisih hasil di bawah ini murni efek kenaikan nominal pinjaman terhadap
kemampuan bayar (DSR) dan kecukupan agunan, bukan karena profil risiko
nasabahnya berubah.

In [21]:
old_features = build_features_from_raw([c["old_application_id"] for c in REPEAT_CASES], profile, slik_full, dhn_full, bank_full, fin_full)

comparison_rows = []
for case, old_row_feat, new_row_feat in zip(REPEAT_CASES, old_features.to_dict("records"), repeat_features.to_dict("records")):
    old_pred = predict_credit_screening(old_row_feat)
    new_pred = predict_credit_screening(new_row_feat)
    for tag, row_feat, pred in [("LAMA", old_row_feat, old_pred), ("BARU", new_row_feat, new_pred)]:
        comparison_rows.append({
            "NIK": row_feat["NIK"], "application_id": row_feat["application_id"], "Pengajuan": tag,
            "Loan Requested": row_feat["loan_requested"], "Collateral Ratio": row_feat["collateral_ratio"],
            "Decision": pred["decision"], "Risk Score": pred["risk_score"], "Zone": pred["zone"],
            "Jenis Kredit": pred["jenis_kredit_rekomendasi"], "Nominal Disetujui": pred["nominal_disetujui"],
            "Jangka Waktu (bulan)": pred["jangka_waktu_bulan"], "Bunga (%)": pred["bunga_persen"],
        })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,NIK,application_id,Pengajuan,Loan Requested,Collateral Ratio,Decision,Risk Score,Zone,Jenis Kredit,Nominal Disetujui,Jangka Waktu (bulan),Bunga (%)
0,3175015609750023,APP202600023,LAMA,750000000,25.00,Layak Bersyarat,0.561,Kuning,KI,750000000,36,12.0
1,3175015609750023,APP202699003,BARU,1200000000,15.62,Layak Bersyarat,0.561,Kuning,KI,1200000000,36,12.0
2,3671016012760030,APP202600030,LAMA,750000000,2.51,Layak Bersyarat,0.667,Kuning,KI,750000000,36,12.0
3,3671016012760030,APP202699004,BARU,1100000000,1.71,Layak Bersyarat,0.667,Kuning,KI,1100000000,36,12.0


In [22]:
for case in REPEAT_CASES:
    nik = repeat_profile[repeat_profile["application_id"] == case["new_application_id"]]["NIK"].iloc[0]
    sub = comparison_df[comparison_df["NIK"] == nik]
    print("=" * 70)
    print(f"NIK {nik}")
    for _, r in sub.iterrows():
        print(f"  [{r['Pengajuan']}] {r['application_id']}: pinjaman {r['Loan Requested']:,} -> "
              f"{r['Decision']} (skor {r['Risk Score']}), disetujui {r['Nominal Disetujui']:,}, "
              f"{r['Jangka Waktu (bulan)']} bulan @ {r['Bunga (%)']}%".replace(",", "."))
print("=" * 70)

NIK 3175015609750023
  [LAMA] APP202600023: pinjaman 750.000.000 -> Layak Bersyarat (skor 0.561). disetujui 750.000.000. 36 bulan @ 12.0%
  [BARU] APP202699003: pinjaman 1.200.000.000 -> Layak Bersyarat (skor 0.561). disetujui 1.200.000.000. 36 bulan @ 12.0%
NIK 3671016012760030
  [LAMA] APP202600030: pinjaman 750.000.000 -> Layak Bersyarat (skor 0.667). disetujui 750.000.000. 36 bulan @ 12.0%
  [BARU] APP202699004: pinjaman 1.100.000.000 -> Layak Bersyarat (skor 0.667). disetujui 1.100.000.000. 36 bulan @ 12.0%


### Kenapa `risk_score`/`decision`/`bunga` PERSIS SAMA walau nominal naik signifikan?

Ini **bukan bug** — verifikasi manual di bawah membuktikannya:

1. **`nominal_disetujui`-nya justru berbeda** (ikut naik sesuai `loan_requested`
   baru) — itu bagian yang memang seharusnya berubah, dan sudah benar
   di tabel di atas.
2. `risk_score` tidak bergeser karena kedua nasabah **jauh
   over-collateralized** di kedua skenario (rasio agunan lama & baru
   sama-sama > 1.5x) — dan formula `collateral_score` Definisi A adalah
   `clip(collateral_ratio / 1.5, 0, 1)`, yang **saturasi di 1.0** begitu
   rasio melewati 1.5x. Model ML mempelajari pola ini dari data training
   (`risk_score` asli juga tidak pernah bergerak lagi begitu agunan
   sudah sangat berlebih) — jadi kenaikan rasio dari 25x→15,6x atau
   2,51x→1,71x sama-sama tetap "penuh aman" secara skor, wajar tidak
   mengubah `risk_score`.
3. `loan_requested` sendiri **tidak pernah jadi bagian dari rumus
   `risk_score`** di Definisi A — nominal pinjaman hanya masuk di
   Layer 2 (policy engine) lewat `nominal_disetujui = min(loan_requested,
   70% nilai agunan)`, bukan komponen skor risiko. Jadi ML yang belajar
   pola ini dari raw features memang seharusnya tidak terlalu sensitif
   terhadap `loan_requested` untuk memprediksi `risk_score` — perilaku
   yang konsisten dengan cara target aslinya dibentuk.

**Kapan nominal_disetujui akan DIPOTONG (bukan disetujui penuh)?** Kalau
`loan_requested` melebihi 70% nilai agunan. Contoh tambahan di bawah:
NIK `3671016012760030` (agunan cuma ~1,88 M) mengajukan **2 Miliar** —
jauh di atas cap 70%-nya (~1,32 M).

In [23]:
cap_test_row = dict(repeat_profile[repeat_profile["application_id"] == "APP202699004"].iloc[0])
cap_test_row["application_id"] = "APP202699005"
cap_test_row["loan_requested"] = 2_000_000_000
cap_test_row["collateral_ratio"] = round(cap_test_row["collateral_market_value"] / 2_000_000_000, 2)

cap_test_profile = pd.DataFrame([cap_test_row])[profile.columns.tolist()]
cap_test_features = build_features_from_raw(["APP202699005"], profile=cap_test_profile,
                                             slik=slik_full, dhn=dhn_full, bank=bank_full, fin=fin_full)
hasil_cap = predict_credit_screening(cap_test_features.iloc[0].to_dict())

collateral_cap = cap_test_row["collateral_market_value"] * 0.7
print(f"loan_requested         : {cap_test_row['loan_requested']:,}".replace(",", "."))
print(f"70% nilai agunan (cap) : {collateral_cap:,.0f}".replace(",", "."))
print(f"nominal_disetujui      : {hasil_cap['nominal_disetujui']:,}".replace(",", "."))
print(f"Decision/Risk Score    : {hasil_cap['decision']} / {hasil_cap['risk_score']}  (masih sama - risk profil nasabah tidak berubah)")
assert hasil_cap["nominal_disetujui"] < cap_test_row["loan_requested"], "Seharusnya nominal_disetujui dipotong oleh cap agunan!"
assert abs(hasil_cap["nominal_disetujui"] - collateral_cap) < 1_000_000, "nominal_disetujui seharusnya = 70% nilai agunan"
print("\nOK - nominal_disetujui benar dipotong ke batas 70% agunan, sementara risk_score/decision tetap mencerminkan profil risiko nasabah (tidak ikut terpotong).")

loan_requested         : 2.000.000.000
70% nilai agunan (cap) : 1.317.220.800
nominal_disetujui      : 1.317.220.800
Decision/Risk Score    : Layak Bersyarat / 0.65  (masih sama - risk profil nasabah tidak berubah)

OK - nominal_disetujui benar dipotong ke batas 70% agunan, sementara risk_score/decision tetap mencerminkan profil risiko nasabah (tidak ikut terpotong).


## Ringkasan

- Input dibangun murni dari `data/raw/retail_customer_profile.csv` + 4
  tabel raw lain lewat `NIK` — join/agregasi disatukan dalam 1 fungsi
  reusable (`build_features_from_raw()`), dipakai konsisten di seluruh
  notebook, sudah tervalidasi 100% cocok dengan `master_dataset.csv`.
- `utils/risk_ml_pipeline.py` (STAGE 1) memakai langsung `identity_agent`,
  `credit_history_agent`, `dhn_agent` dari `utils/agent_pipeline.py` yang
  sudah ada — bukan duplikasi — diverifikasi identik 100% dengan hard-reject
  set training data. `collateral_agent`/`financial_agent` juga dipakai
  ulang untuk insight; Cashflow tetap pakai kalibrasi Definisi A karena versi
  `agent_pipeline.py` terbukti tidak diskriminatif untuk skala data ini.
- **7 kasus diuji end-to-end**: lookup gagal, 3 jenis hard-reject
  (identitas/DHN/SLIK Macet), nasabah baru tanpa riwayat, dan data tidak
  lengkap di tabel lain — semuanya ditangani dengan benar (tidak crash,
  tidak salah klasifikasi) tanpa mengubah 1 baris pun logika di
  `utils/agent_pipeline.py` maupun `utils/risk_ml_pipeline.py`'s STAGE 2/3.